**STEP 1: CHECK THE DATASETS**  
The dataset is the csv of the plan energy

In [ ]:
from datasets import load_dataset

ds = load_dataset("hoangphuc090104/DCR_Energy_Plan")["train"]

# Số dòng
num_rows = ds.num_rows

# Số cột
num_columns = len(ds.column_names)

# Kiểu dữ liệu từng cột
column_types = ds.features

# Cột có giá trị NULL
null_columns = []
for col in ds.column_names:
    # đếm số phần tử None / Null
    null_count = sum(1 for v in ds[col] if v is None)
    if null_count > 0:
        null_columns.append((col, null_count))

print("Rows:", num_rows)
print("Columns:", num_columns)
print("Column Types:", column_types)
print("Columns containing NULL values:", null_columns)

Rows: 25670
Columns: 25
Column Types: {'id': Value('string'), 'plan_id': Value('string'), 'type': Value('string'), 'brand': Value('string'), 'brand_name': Value('string'), 'display_name': Value('string'), 'fuel_type': Value('string'), 'customer_type': Value('string'), 'factsheet_url': Value('string'), 'effective_from': Value('string'), 'effective_to': Value('string'), 'last_updated': Value('string'), 'geography': Value('string'), 'electricity_contract': Value('string'), 'pcr': Value('string'), 'compliance_score': Value('float64'), 'compliance_checklist': Value('string'), 'metadata': Value('string'), 'created_at': Value('string'), 'updated_at': Value('string'), 'created_by': Value('float64'), 'updated_by': Value('float64'), 'postcode': Value('float64'), 'ops_status': Value('string'), 'reference_price': Value('string')}
Columns containing NULL values: [('factsheet_url', 3565), ('effective_to', 24035), ('pcr', 1663), ('compliance_score', 25670), ('created_by', 25670), ('updated_by', 25670

In [ ]:

# Remove the NULL columns
null_cols = []
for col in ds.column_names:
    if any(v is None for v in ds[col]):
        null_cols.append(col)
ds = ds.remove_columns(null_cols)

# Remove the single value column
single_value_cols = []
for col in ds.column_names:
    if len(set(ds[col])) == 1:
        single_value_cols.append(col)
ds = ds.remove_columns(single_value_cols)

# Remove the meaningless column (with contain id or datetime - model cant understand it well)
meaningless_cols = [
    "id", "plan_id", "type", "brand", "factsheet_url", "effective_from",
    "last_updated", "geography", "metadata", "created_at", "updated_at",
    "created_by", "postcode", "ops_status", "text"
]
drop_cols = [c for c in meaningless_cols if c in ds.column_names]
ds = ds.remove_columns(drop_cols)


import json
import re

def normalize_and_textify(example):
    out = {}

    for k, v in example.items():
        if isinstance(v, str):
            s = v.strip()
            parsed = None
            if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
                try:
                    parsed = json.loads(s)
                except:
                    parsed = None

            if parsed is not None:
                v = parsed

        text = str(v)
        text = re.sub(r"[{}\[\]\"']", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        out[k] = text

    return {"text": " ".join(f"{k}: {v}" for k, v in out.items())}


ds = ds.map(normalize_and_textify)





Map:   0%|          | 0/25670 [00:00<?, ? examples/s]

In [ ]:
print(len(ds[0]['text']))

2985


**2. READY FOR SOME SAMPLE EMBEDDING!**

In [ ]:
pip install langchain-huggingface

In [ ]:
print(type(ds[0]["text"]))

<class 'dict'>


**a) Sentence-transformers: BAAI/bge-m3**
(Embedding 5 plan for example)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

subset = [ds[i]["text"]  for i in range(5)]
vectors = model.embed_documents(subset)

for i, vec in enumerate(vectors):
    print(f"Embedding {i}: length={len(vec)}")
    print(vec)
    print()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding 0: length=1024
[-0.014001037925481796, 0.004196053370833397, -0.041061513125896454, -0.030664687976241112, -0.0047417557798326015, -0.06474486738443375, 0.030108995735645294, -0.0034426236525177956, 0.012677740305662155, 0.03811792656779289, 0.042104147374629974, -0.011880646459758282, -0.002857237821444869, -0.03145446628332138, -0.017378218472003937, -0.015922153368592262, -0.0237748920917511, 0.012141949497163296, 0.030259625986218452, 0.01030126865953207, 0.015896854922175407, -0.0005134166567586362, 0.025390971451997757, 0.0038326019421219826, 0.03239816054701805, 0.06389391422271729, -0.043919652700424194, -0.011645456776022911, 0.05694755166769028, 0.05309591069817543, 0.012358998879790306, -0.009072533808648586, 0.050292737782001495, -0.03558182343840599, -0.024814188480377197, 0.00706954812631011, -0.0014849090948700905, 0.0036614027339965105, -0.048771943897008896, 0.0118490532040596, 0.016721097752451897, -0.014060777612030506, -0.023802796378731728, 0.001292729983

**b) Sentence-transformers: Alibaba-NLP/gte-modernbert-base** (Embedding 5 plan for example)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(
    model_name="Alibaba-NLP/gte-modernbert-base",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={}
)

subset = [ds[i]["text"] for i in range(5)]
vectors = model.embed_documents(subset)

for i, vec in enumerate(vectors):
    print(f"Embedding {i}: length={len(vec)}")
    print(vec)
    print()

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Embedding 0: length=768
[0.1632080078125, 0.10528564453125, -0.912109375, -1.2001953125, 1.5458984375, 1.2861328125, 0.8896484375, 0.277099609375, 0.09576416015625, -1.4931640625, 0.92822265625, -1.138671875, -0.95849609375, 0.461669921875, -1.49609375, 0.84521484375, 0.70849609375, 2.041015625, 2.677734375, 0.294677734375, 0.6884765625, -0.203125, -1.4833984375, -1.1201171875, 1.380859375, -1.1181640625, -0.1949462890625, 0.52392578125, -0.9814453125, -0.31396484375, 0.24560546875, -2.712890625, 1.517578125, -1.8134765625, 1.6083984375, -0.6640625, 1.689453125, -1.908203125, -1.0458984375, -1.2373046875, -0.95556640625, -3.955078125, 0.65576171875, -0.0289764404296875, -0.9208984375, -1.015625, -0.66650390625, -0.93212890625, 0.78955078125, 1.412109375, 0.5302734375, -0.51904296875, 2.513671875, 0.06146240234375, 0.2266845703125, -1.2626953125, 0.5517578125, 2.36328125, -0.5888671875, -0.798828125, 0.314697265625, -0.6708984375, -1.3291015625, -1.0029296875, -0.81396484375, 0.42382812

**c) Sentence-transformers: embaas/sentence-transformers-e5-large-v2** (Embedding 5 plan for example)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(
    model_name="embaas/sentence-transformers-e5-large-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Với E5, nên thêm prefix "passage: " hoặc "query: "
subset = [ "passage: " + ds[i]["text"] for i in range(5) ]

vectors = model.embed_documents(subset)

for i, vec in enumerate(vectors):
    print(f"Embedding {i}: length={len(vec)}")
    print(vec)
    print()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: embaas/sentence-transformers-e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 0: length=1024
[0.00524113979190588, -0.04103041812777519, 0.0453227199614048, 0.00580157246440649, -0.03529423475265503, 0.01710616983473301, -0.04267733171582222, -0.01626257225871086, -0.02793898992240429, 0.0022520972415804863, 0.04947448894381523, -0.009427497163414955, 0.017103934660553932, 0.022283319383859634, 0.021139470860362053, 0.03861815482378006, 0.01696019433438778, -0.0005538788973353803, 0.04767005145549774, -0.056078992784023285, 0.014773690141737461, -0.05416688695549965, -0.0012581608025357127, 0.018471574410796165, -0.013191933743655682, -0.0392904169857502, -0.011037937365472317, -0.016545837745070457, -0.022005539387464523, 0.020551344379782677, 0.010474098846316338, -0.012204199098050594, 0.005313688889145851, -0.029893167316913605, 0.04686548933386803, 0.013411104679107666, -0.04615883156657219, -0.057410530745983124, 0.008575227111577988, -0.014645257033407688, -0.001531639019958675, 0.05369309335947037, -0.026244934648275375, 0.023957043886184692, 0

**PREPARE DATASETS FOR BENCHMARK**


TESTING:
If two plans have similarity (like same retailer, same something,...) thier vector have the high cosine similarity too?
And if two plan different, thier cosine similarity in the plan will not very high?

**TEST BY PLAN BRAND NAME**

In [ ]:
from datasets import load_dataset
from collections import defaultdict

data = load_dataset("hoangphuc090104/DCR_Energy_Plan")["train"]

# Remove NULL columns
null_cols = [c for c in data.column_names if any(v is None for v in data[c])]
data = data.remove_columns(null_cols)

# Remove single-value columns
single_value_cols = [c for c in data.column_names if len(set(data[c])) == 1]
data = data.remove_columns(single_value_cols)

# Remove meaningless columns
meaningless_cols = [
    "id", "plan_id", "type", "brand", "factsheet_url", "effective_from",
    "last_updated", "metadata", "created_at", "updated_at",
    "created_by", "postcode", "ops_status", "reference_price"
]

drop_cols = [c for c in meaningless_cols if c in data.column_names]
data = data.remove_columns(drop_cols)

print("Done. All meaningless columns dropped.")

# Show remaining columns
print("Remaining columns:")
print(len(data.column_names))
for col in data.column_names:
    print("-", col)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


energy_plans.csv:   0%|          | 0.00/212M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25670 [00:00<?, ? examples/s]

Done. All meaningless columns dropped.
Remaining columns:
5
- brand_name
- display_name
- customer_type
- geography
- electricity_contract


In [ ]:
import json


def clean_contract(contract):
    # 1) Bỏ isFixed
    contract.pop("isFixed", None)

    # 2) discounts: bỏ type, methodUType
    if isinstance(contract.get("discounts"), list):
        for d in contract["discounts"]:
            if isinstance(d, dict):
                d.pop("type", None)
                d.pop("methodUType", None)

    # 3) tariffPeriod cleanup
    if isinstance(contract.get("tariffPeriod"), list):
        for tp in contract["tariffPeriod"]:
            if not isinstance(tp, dict):
                continue

            # singleRate
            sr = tp.get("singleRate")
            if isinstance(sr, dict):
                sr.pop("period", None)
                sr.pop("description", None)
                sr.pop("displayName", None)

            tp.pop("displayName", None)
            tp.pop("rateBlockUType", None)
            tp.pop("dailySupplyChargeType", None)

    # 4) fees cleanup
    if isinstance(contract.get("fees"), list):
        cleaned_list = []
        for fee in contract["fees"]:
            if isinstance(fee, dict):
                cleaned_fee = {
                    k: v for k, v in fee.items()
                    if k not in ["term", "type"]
                }
                cleaned_list.append(cleaned_fee)
        contract["fees"] = cleaned_list

    # 5) solarFeedInTariff cleanup
    if isinstance(contract.get("solarFeedInTariff"), list):
        for s in contract["solarFeedInTariff"]:
            if not isinstance(s, dict):
                continue

            s.pop("scheme", None)
            s.pop("payerType", None)
            s.pop("tariffUType", None)

            st = s.get("singleTariff")
            if isinstance(st, dict):
                st.pop("period", None)

                rates = st.get("rates")
                if isinstance(rates, list):
                    for r in rates:
                        if isinstance(r, dict):
                            r.pop("measureUnit", None)

    return contract

def process_row(row):
    contract = json.loads(row["electricity_contract"])
    cleaned_con = clean_contract(contract)

    geography = json.loads(row["geography"])

    return {
        "distributors": geography["distributors"],
        "electricity_contract": json.dumps(cleaned_con)
    }

data = data.map(process_row)
data = data.remove_columns(["geography"])

# Kiểm tra 1 phần tử sau khi xử lý
print(data[0])
print(json.dumps(json.loads(data[0]["electricity_contract"]), indent=2))


Map:   0%|          | 0/25670 [00:00<?, ? examples/s]

{'brand_name': '1st Energy', 'display_name': '1st Blue 22 - Single Rate', 'customer_type': 'RESIDENTIAL', 'electricity_contract': '{"fees": [{"amount": "16.5", "description": "Connection fee (inc. GST), generally applies for move-in requests"}, {"amount": "16.5", "description": "Disconnection fee (inc. GST), generally applies for move-out requests"}, {"amount": "15.00", "description": "Cheque dishonour payment fee (inc. GST)"}, {"amount": "7.5", "description": "Direct debit dishonour payment fee (inc. GST)"}], "terms": "This offer is available for customers currently taking supply on a corresponding single rate network tariff. For further details on the information presented in this fact sheet, or the terms and conditions of the offer, please visit www.1stenergy.com.au or call 1st Energy on 1300 426 594.", "termType": "ONGOING", "discounts": [{"description": "22% guaranteed discount off the market contract usage and supply rates.", "displayName": "Guaranteed discount off usage and supp

In [ ]:
# Brand filter
brand_counts = defaultdict(int)
for row in data:
    brand = row.get("brand_name", None)
    if brand:
        brand_counts[brand] += 1

eligible_brands = [b for b, c in brand_counts.items() if c >= 101]

print("Branch that have 101 elements (able to be chosen):", len(eligible_brands))
print("Branch list")

for i in range(0, 11):
    print(f"-{eligible_brands[i]}: {brand_counts[eligible_brands[i]]}")

Branch that have 101 elements (able to be chosen): 28
Branch list
-1st Energy: 446
-ActewAGL: 262
-AGL: 1953
-Alinta Energy: 651
-Amber Electric: 701
-Arcline by RACV - Energy: 168
-Blue NRG: 361
-CovaU: 376
-Diamond Energy: 211
-Dodo Power & Gas: 125
-EnergyAustralia: 1312


**SEMATIC SIMILARITY TEST 1: BRAND NAME SIMILARITY**

In [ ]:
from collections import defaultdict
import random

# Step 1: pick first 10 eligible brands
selected_brands = eligible_brands[:5]

# Prepare containers
query_data = []
testing_data = []

# Step 2: iterate through dataset and collect rows
brand_buckets = {b: [] for b in selected_brands}

for row in data:
    b = row.get("brand_name", None)
    if b in brand_buckets:
        brand_buckets[b].append(row)

# Step 3: extract 1 query + 100 testing rows for each brand
for b in selected_brands:
    rows = brand_buckets[b]

    # safety check
    if len(rows) < 10:
        continue

    # deterministic first-row selection, or shuffle then pick
    # random.shuffle(rows)

    query_data.append(rows[0])      # 1 query
    testing_data.extend(rows[1:11])  # 100 test rows

# Step 4: shuffle testing dataset
random.shuffle(testing_data)

# Final debug prints
print("Selected brands:", selected_brands)
print("Query count:", len(query_data))
print("Testing count:", len(testing_data))
print("Example query row:")
print(query_data[0])
print("Example testing row:")
print(testing_data[0])
print(selected_brands)

Selected brands: ['1st Energy', 'ActewAGL', 'AGL', 'Alinta Energy', 'Amber Electric']
Query count: 5
Testing count: 50
Example query row:
{'brand_name': '1st Energy', 'display_name': '1st Blue 22 - Single Rate', 'customer_type': 'RESIDENTIAL', 'electricity_contract': '{"fees": [{"amount": "16.5", "description": "Connection fee (inc. GST), generally applies for move-in requests"}, {"amount": "16.5", "description": "Disconnection fee (inc. GST), generally applies for move-out requests"}, {"amount": "15.00", "description": "Cheque dishonour payment fee (inc. GST)"}, {"amount": "7.5", "description": "Direct debit dishonour payment fee (inc. GST)"}], "terms": "This offer is available for customers currently taking supply on a corresponding single rate network tariff. For further details on the information presented in this fact sheet, or the terms and conditions of the offer, please visit www.1stenergy.com.au or call 1st Energy on 1300 426 594.", "termType": "ONGOING", "discounts": [{"descr

In [ ]:
from datasets import Dataset

# Convert testing_data list → HF Dataset
testing_ds = Dataset.from_list(testing_data)

# Normalize
testing_ds = testing_ds.map(normalize_and_textify)

# Same with query
query_ds = Dataset.from_list(query_data)
query_ds = query_ds.map(normalize_and_textify)

print(type(testing_ds[0]["text"]))
print(type(query_ds[0]["text"] ))

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

<class 'str'>
<class 'str'>


In [ ]:
testing_ds[0]["brand_name"]
query_ds[1]["brand_name"]
testing_ds[0]["text"]

'brand_name: AGL display_name: Residential Netflix Plan - 3rd Party customer_type: RESIDENTIAL electricity_contract: fees: amount: 66.69, description: Fee may be charged when reconnecting or reading your meter when you move into a property or change retailer. Includes GST. Fees may vary.\\nAdditional fees and charges may apply. Please see the standing offer terms and conditions at agl.com.au, amount: 95.65, description: Fee may be charged when disconnecting or reading your meter when you move out of a property or change retailer. Includes GST. Fees may vary., rate: 0.0054, description: The amount is GST inclusive and applies to card payments made at Australia Post outlets, rate: 0.0015, description: The amount is GST inclusive and applies to payments made by Visa debit cards., rate: 0.0065, description: The amount is GST inclusive and applies to payments made by Visa credit cards., rate: 0.0075, description: The amount is GST inclusive and applies to payments made by Mastercard credit 

**Test for e5-large Embedding Model**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(
    model_name="embaas/sentence-transformers-e5-large-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Thêm prefix bắt buộc cho e5
testing_texts = ["passage: " + t for t in testing_ds["text"]]

test_vectors_e5 = model.embed_documents(testing_texts)

print(f"Total testing samples: {len(test_vectors_e5)}")
print(f"Embedding vector length: {len(test_vectors_e5[0])}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: embaas/sentence-transformers-e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total testing samples: 50
Embedding vector length: 1024


In [ ]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

# Hàm tính cosine similarity
def cosine_sim(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = []

for i in range(len(query_ds)):
    query_text = query_ds[i]["text"]
    query_brand = query_ds[i]["brand_name"]

    # Embed query
    q_vec = model.embed_query(query_text)

    # Tính similarity với toàn bộ test_vectors
    sims = []
    for idx, v in enumerate(test_vectors_e5):
        sim = cosine_sim(q_vec, v)
        sims.append((sim, idx))

    # Sort tăng dần → chọn 5 giá trị nhỏ nhất
    sims_sorted = sorted(sims, key=lambda x: x[0], reverse=True)
    print(sims_sorted[:5])

    top5 = sims_sorted[:5]

    # Kiểm tra brand_name
    correct = 0
    print("Brand")
    for sim, idx in top5:
        test_brand = testing_ds[idx]["brand_name"]
        print(test_brand)
        if test_brand == query_brand:
            correct += 1

    score = correct / 5
    scores.append(score)

# Điểm trung bình
avg_score = sum(scores) / len(scores)
print("Average score:", avg_score)

[(np.float64(0.9941099349320187), 32), (np.float64(0.9828571553354425), 27), (np.float64(0.9822447832488111), 10), (np.float64(0.9821460437340638), 8), (np.float64(0.9818003354689722), 3)]
Brand
1st Energy
1st Energy
1st Energy
1st Energy
1st Energy
[(np.float64(0.9827507797231072), 49), (np.float64(0.979136993122894), 18), (np.float64(0.978668715104045), 39), (np.float64(0.9709633594776713), 25), (np.float64(0.9692267081697045), 0)]
Brand
ActewAGL
ActewAGL
ActewAGL
ActewAGL
ActewAGL
[(np.float64(0.9965900418636471), 42), (np.float64(0.995949911776919), 20), (np.float64(0.995949911776919), 21), (np.float64(0.994928966868931), 6), (np.float64(0.994928966868931), 7)]
Brand
AGL
AGL
AGL
AGL
AGL
[(np.float64(0.9913104198867534), 28), (np.float64(0.9889466513401803), 13), (np.float64(0.9886224818815397), 48), (np.float64(0.9872062320031857), 34), (np.float64(0.9863931113005087), 41)]
Brand
Alinta Energy
Alinta Energy
Alinta Energy
Alinta Energy
Alinta Energy
[(np.float64(0.9899418126777196),

**BGE-M3**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Không cần prefix
testing_texts = testing_ds["text"]

test_vectors = model.embed_documents(testing_texts)

print("Total testing samples:", len(test_vectors))
print("Embedding vector length:", len(test_vectors[0]))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Total testing samples: 50
Embedding vector length: 1024


In [ ]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

# Hàm tính cosine similarity
def cosine_sim(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = []

for i in range(len(query_ds)):
    query_text = query_ds[i]["text"]
    query_brand = query_ds[i]["brand_name"]

    # Embed query
    q_vec = model.embed_query(query_text)

    # Tính similarity với toàn bộ test_vectors
    sims = []
    for idx, v in enumerate(test_vectors):
        sim = cosine_sim(q_vec, v)
        sims.append((sim, idx))

    # Sort tăng dần → chọn 5 giá trị nhỏ nhất
    sims_sorted = sorted(sims, key=lambda x: x[0], reverse=True)
    print(sims_sorted[:5])

    top5 = sims_sorted[:5]

    # Kiểm tra brand_name
    correct = 0
    print("Brand")
    for sim, idx in top5:
        test_brand = testing_ds[idx]["brand_name"]
        print(test_brand)
        if test_brand == query_brand:
            correct += 1

    score = correct / 5
    scores.append(score)

# Điểm trung bình
avg_score = sum(scores) / len(scores)
print("Average score:", avg_score)

[(np.float64(0.9970442842153621), 32), (np.float64(0.9770360876058365), 8), (np.float64(0.9761733137927816), 27), (np.float64(0.9742668898245682), 3), (np.float64(0.9723439044485688), 4)]
Brand
1st Energy
1st Energy
1st Energy
1st Energy
1st Energy
[(np.float64(0.9748608430477554), 49), (np.float64(0.9682835434106839), 18), (np.float64(0.9517060547077272), 33), (np.float64(0.9510911737232155), 35), (np.float64(0.9470100506165189), 39)]
Brand
ActewAGL
ActewAGL
ActewAGL
ActewAGL
ActewAGL
[(np.float64(0.9984402129875759), 16), (np.float64(0.9984293692667672), 21), (np.float64(0.9982517105992151), 47), (np.float64(0.9975319328349511), 7), (np.float64(0.9944237426321197), 45)]
Brand
AGL
AGL
AGL
AGL
AGL
[(np.float64(0.9960922002274752), 34), (np.float64(0.9953776041522552), 28), (np.float64(0.986815063761708), 48), (np.float64(0.9867800144891931), 13), (np.float64(0.9698929739167167), 36)]
Brand
Alinta Energy
Alinta Energy
Alinta Energy
Alinta Energy
Alinta Energy
[(np.float64(0.991457804419

**Alibaba-NLP/gte-modernbert-base**

In [ ]:
print(len(testing_ds))

20


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(
    model_name="Alibaba-NLP/gte-modernbert-base",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Không cần prefix
testing_texts = testing_ds["text"]

test_vectors_modernbert = model.embed_documents(testing_texts)

print("Total testing samples:", len(test_vectors_modernbert))
print("Embedding vector length:", len(test_vectors_modernbert[0]))

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Total testing samples: 50
Embedding vector length: 768


In [ ]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

# Hàm tính cosine similarity
def cosine_sim(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = []

for i in range(len(query_ds)):
    query_text = query_ds[i]["text"]
    query_brand = query_ds[i]["brand_name"]

    # Embed query
    q_vec = model.embed_query(query_text)

    # Tính similarity với toàn bộ test_vectors
    sims = []
    for idx, v in enumerate(test_vectors_modernbert):
        sim = cosine_sim(q_vec, v)
        sims.append((sim, idx))

    # Sort tăng dần → chọn 5 giá trị nhỏ nhất
    sims_sorted = sorted(sims, key=lambda x: x[0], reverse=True)
    print(sims_sorted[:5])

    top5 = sims_sorted[:5]

    # Kiểm tra brand_name
    correct = 0
    print("Brand")
    for sim, idx in top5:
        test_brand = testing_ds[idx]["brand_name"]
        print(test_brand)
        if test_brand == query_brand:
            correct += 1

    score = correct / 5
    scores.append(score)

# Điểm trung bình
avg_score = sum(scores) / len(scores)
print("Average score:", avg_score)

[(np.float64(0.9996026892365973), 32), (np.float64(0.9796479266454077), 8), (np.float64(0.9788272217873484), 27), (np.float64(0.9753000657075512), 4), (np.float64(0.974737249710542), 14)]
Brand
1st Energy
1st Energy
1st Energy
1st Energy
1st Energy
[(np.float64(0.9678595383927261), 49), (np.float64(0.9635527413339943), 18), (np.float64(0.943411303644431), 35), (np.float64(0.935189767628698), 33), (np.float64(0.9250268710239522), 1)]
Brand
ActewAGL
ActewAGL
ActewAGL
ActewAGL
ActewAGL
[(np.float64(0.9989085755187846), 7), (np.float64(0.9982609630607001), 21), (np.float64(0.9977862462786472), 16), (np.float64(0.9968578673006118), 47), (np.float64(0.9944858738186864), 42)]
Brand
AGL
AGL
AGL
AGL
AGL
[(np.float64(0.9953816085941922), 34), (np.float64(0.9937575010617843), 28), (np.float64(0.9847483955908585), 13), (np.float64(0.9847359954089635), 48), (np.float64(0.9734221707870337), 36)]
Brand
Alinta Energy
Alinta Energy
Alinta Energy
Alinta Energy
Alinta Energy
[(np.float64(0.98718160435640

**TEST 2: Triplet Evaluation**

In [ ]:
import json
import ast
from collections import Counter, defaultdict

def parse_distributors(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except:
            pass

        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return parsed
        except:
            pass

        return [value]

    return []

def parse_contract(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except:
            return {}

    return {}

def has_solar_feed_in(contract):
    solar = contract.get("solarFeedInTariff")
    return isinstance(solar, list) and len(solar) > 0

def has_demand_charges(contract):
    tariff_periods = contract.get("tariffPeriod")

    if not isinstance(tariff_periods, list):
        return False

    for period in tariff_periods:
        if not isinstance(period, dict):
            continue

        demand_charges = period.get("demandCharges")
        if isinstance(demand_charges, list) and len(demand_charges) > 0:
            return True

    return False

customer_type_counter = Counter()
pricing_model_counter = Counter()
distributor_counter = Counter()
solar_counter = Counter()
demand_counter = Counter()
signature_counter = Counter()

for row in data:
    customer_type = row.get("customer_type")

    distributors = parse_distributors(row.get("distributors"))
    contract = parse_contract(row.get("electricity_contract"))

    pricing_model = contract.get("pricingModel")
    solar_flag = has_solar_feed_in(contract)
    demand_flag = has_demand_charges(contract)

    customer_type_counter[customer_type] += 1
    pricing_model_counter[pricing_model] += 1
    solar_counter[solar_flag] += 1
    demand_counter[demand_flag] += 1

    for distributor in distributors:
        distributor_counter[distributor] += 1

        signature = (
            customer_type,
            pricing_model,
            distributor,
            solar_flag,
            demand_flag
        )
        signature_counter[signature] += 1

print("Customer types:")
for k, v in customer_type_counter.most_common():
    print(f"- {k}: {v}")

print("\nPricing models:")
for k, v in pricing_model_counter.most_common():
    print(f"- {k}: {v}")

print("\nDistributors:")
for k, v in distributor_counter.most_common():
    print(f"- {k}: {v}")

print("\nHas solar feed-in tariff:")
for k, v in solar_counter.most_common():
    print(f"- {k}: {v}")

print("\nHas demand charges:")
for k, v in demand_counter.most_common():
    print(f"- {k}: {v}")

print("\nTop 5 signatures:")
for sig, count in signature_counter.most_common(5):
    print(f"- {sig}: {count}")

Customer types:
- RESIDENTIAL: 16966
- BUSINESS: 8704

Pricing models:
- TIME_OF_USE_CONT_LOAD: 8164
- SINGLE_RATE_CONT_LOAD: 6149
- TIME_OF_USE: 5785
- SINGLE_RATE: 5235
- FLEXIBLE: 187
- FLEXIBLE_CONT_LOAD: 150

Distributors:
- Ausgrid: 3281
- Essential Energy Standard: 3170
- Energex: 2842
- Essential Energy Far West: 2699
- Essential Energy: 2673
- Endeavour: 2658
- Powercor: 2247
- Citipower: 2234
- AusNet Services (electricity): 2177
- Jemena: 2058
- United Energy: 2057
- SA Power Networks: 1376
- Evoenergy: 1074
- Ergon: 233
- TasNetworks: 211

Has solar feed-in tariff:
- True: 25051
- False: 619

Has demand charges:
- False: 21537
- True: 4133

Top 5 signatures:
- ('RESIDENTIAL', 'TIME_OF_USE_CONT_LOAD', 'Essential Energy Standard', True, False): 935
- ('RESIDENTIAL', 'TIME_OF_USE_CONT_LOAD', 'Essential Energy', True, False): 819
- ('RESIDENTIAL', 'TIME_OF_USE_CONT_LOAD', 'Essential Energy Far West', True, False): 816
- ('RESIDENTIAL', 'TIME_OF_USE_CONT_LOAD', 'Ausgrid', True, 

In [ ]:
import json
import ast
import random
from collections import defaultdict

random.seed(42)

SIGNATURE_FIELDS = [
    "customer_type",
    "pricingModel",
    "distributor",
    "has_solarFeedInTariff",
    "has_demandCharges"
]

def parse_distributors(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except:
            pass

        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return parsed
        except:
            pass

        return [value]

    return []

def parse_contract(value):
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except:
            return {}

    return {}

def has_solar_feed_in(contract):
    solar = contract.get("solarFeedInTariff")
    return isinstance(solar, list) and len(solar) > 0

def has_demand_charges(contract):
    tariff_periods = contract.get("tariffPeriod")

    if not isinstance(tariff_periods, list):
        return False

    for period in tariff_periods:
        if not isinstance(period, dict):
            continue

        demand_charges = period.get("demandCharges")
        if isinstance(demand_charges, list) and len(demand_charges) > 0:
            return True

    return False

def row_to_text(row):
    if "text" in row and isinstance(row["text"], str):
        return row["text"]

    parts = []
    for k, v in row.items():
        if k in ["electricity_contract", "distributors"]:
            parts.append(f"{k}: {v}")
        elif isinstance(v, (str, int, float, bool)):
            parts.append(f"{k}: {v}")

    text = " | ".join(parts)
    return text

def get_plan_items(data):
    items = []

    for idx, row in enumerate(data):
        contract = parse_contract(row.get("electricity_contract"))
        distributors = parse_distributors(row.get("distributors"))

        customer_type = row.get("customer_type")
        pricing_model = contract.get("pricingModel")
        solar_flag = has_solar_feed_in(contract)
        demand_flag = has_demand_charges(contract)
        brand = row.get("brand_name")

        for distributor in distributors:
            signature = (
                customer_type,
                pricing_model,
                distributor,
                solar_flag,
                demand_flag
            )

            items.append({
                "idx": idx,
                "id": row.get("id", str(idx)),
                "brand_name": brand,
                "signature": signature,
                "row": row,
                "text": row_to_text(row)
            })

    return items

def diff_signature_count(sig1, sig2):
    return sum(a != b for a, b in zip(sig1, sig2))

def diff_signature_field(sig1, sig2):
    diffs = []

    for field, a, b in zip(SIGNATURE_FIELDS, sig1, sig2):
        if a != b:
            diffs.append((field, a, b))

    return diffs

items = get_plan_items(data)

signature_groups = defaultdict(list)
brand_signature_groups = defaultdict(list)

for item in items:
    signature_groups[item["signature"]].append(item)
    brand_signature_groups[(item["brand_name"], item["signature"])].append(item)

triplets = []
used_anchor_ids = set()

for anchor in items:
    if anchor["id"] in used_anchor_ids:
        continue

    anchor_sig = anchor["signature"]
    anchor_brand = anchor["brand_name"]

    positive_candidates = [
        x for x in brand_signature_groups[(anchor_brand, anchor_sig)]
        if x["idx"] != anchor["idx"]
    ]

    if not positive_candidates:
        positive_candidates = [
            x for x in signature_groups[anchor_sig]
            if x["idx"] != anchor["idx"]
        ]

    if not positive_candidates:
        continue

    negative_candidates_same_brand = [
        x for x in items
        if x["idx"] != anchor["idx"]
        and x["brand_name"] == anchor_brand
        and diff_signature_count(anchor_sig, x["signature"]) == 1
    ]

    if negative_candidates_same_brand:
        negative_candidates = negative_candidates_same_brand
    else:
        negative_candidates = [
            x for x in items
            if x["idx"] != anchor["idx"]
            and diff_signature_count(anchor_sig, x["signature"]) == 1
        ]

    if not negative_candidates:
        continue

    positive = random.choice(positive_candidates)
    negative = random.choice(negative_candidates)

    diffs = diff_signature_field(anchor_sig, negative["signature"])
    diff_field, anchor_value, negative_value = diffs[0]

    reason = (
        f"anchor and positive have the same signature; "
        f"negative differs only by {diff_field}: "
        f"anchor={anchor_value}, negative={negative_value}"
    )

    triplets.append({
        "anchor_id": anchor["id"],
        "positive_id": positive["id"],
        "negative_id": negative["id"],
        "anchor_text": anchor["text"],
        "positive_text": positive["text"],
        "negative_text": negative["text"],
        "reason": reason
    })

    used_anchor_ids.add(anchor["id"])

    if len(triplets) >= 10:
        break

print("Created triplet:", len(triplets))

Created triplet: 10


In [ ]:
for i, t in enumerate(triplets):
    print(f"Triplet {i + 1}: {t['reason']}")

Triplet 1: anchor and positive have the same signature; negative differs only by pricingModel: anchor=SINGLE_RATE, negative=TIME_OF_USE_CONT_LOAD
Triplet 2: anchor and positive have the same signature; negative differs only by distributor: anchor=Ausgrid, negative=TasNetworks
Triplet 3: anchor and positive have the same signature; negative differs only by distributor: anchor=Ausgrid, negative=United Energy
Triplet 4: anchor and positive have the same signature; negative differs only by distributor: anchor=Ausgrid, negative=Endeavour
Triplet 5: anchor and positive have the same signature; negative differs only by pricingModel: anchor=TIME_OF_USE, negative=TIME_OF_USE_CONT_LOAD
Triplet 6: anchor and positive have the same signature; negative differs only by pricingModel: anchor=TIME_OF_USE_CONT_LOAD, negative=SINGLE_RATE_CONT_LOAD
Triplet 7: anchor and positive have the same signature; negative differs only by distributor: anchor=Ausgrid, negative=Endeavour
Triplet 8: anchor and positive

**E5 Model Similarity Semnatic**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

model = HuggingFaceEmbeddings(
    model_name="embaas/sentence-transformers-e5-large-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

def cosine_sim(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

anchor_texts = ["query: " + t["anchor_text"] for t in triplets]
positive_texts = ["passage: " + t["positive_text"] for t in triplets]
negative_texts = ["passage: " + t["negative_text"] for t in triplets]

anchor_vectors = model.embed_documents(anchor_texts)
positive_vectors = model.embed_documents(positive_texts)
negative_vectors = model.embed_documents(negative_texts)

correct = 0
total = len(triplets)

for i in range(total):
    sim_positive = cosine_sim(anchor_vectors[i], positive_vectors[i])
    sim_negative = cosine_sim(anchor_vectors[i], negative_vectors[i])

    if sim_positive > sim_negative:
        correct += 1

triplet_accuracy = correct / total if total > 0 else 0

print("Triplet Evaluation Result")
print("Total triplets:", total)
print("Correct triplets:", correct)
print("Triplet accuracy:", triplet_accuracy)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: embaas/sentence-transformers-e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Triplet Evaluation Result
Total triplets: 10
Correct triplets: 7
Triplet accuracy: 0.7


BGE M3

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

def cosine_sim(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

anchor_texts = [t["anchor_text"] for t in triplets]
positive_texts = [t["positive_text"] for t in triplets]
negative_texts = [t["negative_text"] for t in triplets]

anchor_vectors = model.embed_documents(anchor_texts)
positive_vectors = model.embed_documents(positive_texts)
negative_vectors = model.embed_documents(negative_texts)

correct = 0
total = len(triplets)

for i in range(total):
    sim_positive = cosine_sim(anchor_vectors[i], positive_vectors[i])
    sim_negative = cosine_sim(anchor_vectors[i], negative_vectors[i])

    if sim_positive > sim_negative:
        correct += 1

triplet_accuracy = correct / total if total > 0 else 0

print("Triplet Evaluation Result")
print("Model: BAAI/bge-m3")
print("Total triplets:", total)
print("Correct triplets:", correct)
print("Triplet accuracy:", triplet_accuracy)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Triplet Evaluation Result
Model: BAAI/bge-m3
Total triplets: 10
Correct triplets: 9
Triplet accuracy: 0.9


Modernbert

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

model = HuggingFaceEmbeddings(
    model_name="Alibaba-NLP/gte-modernbert-base",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

def cosine_sim(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

anchor_texts = [t["anchor_text"] for t in triplets]
positive_texts = [t["positive_text"] for t in triplets]
negative_texts = [t["negative_text"] for t in triplets]

anchor_vectors = model.embed_documents(anchor_texts)
positive_vectors = model.embed_documents(positive_texts)
negative_vectors = model.embed_documents(negative_texts)

correct = 0
total = len(triplets)

for i in range(total):
    sim_positive = cosine_sim(anchor_vectors[i], positive_vectors[i])
    sim_negative = cosine_sim(anchor_vectors[i], negative_vectors[i])

    if sim_positive > sim_negative:
        correct += 1

triplet_accuracy = correct / total if total > 0 else 0

print("Triplet Evaluation Result")
print("Model: Alibaba-NLP/gte-modernbert-base")
print("Total triplets:", total)
print("Correct triplets:", correct)
print("Triplet accuracy:", triplet_accuracy)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Triplet Evaluation Result
Model: Alibaba-NLP/gte-modernbert-base
Total triplets: 10
Correct triplets: 10
Triplet accuracy: 1.0
